# Component 1 - improved training runs (v12 candidates)

Goal: improve the deployed v10 model (test macro-F1 0.747; weakest: **S2R** F1 0.70 and **EB** F1 0.73, with the
Expected-Behaviour phrasing sensitivity documented in Section 5.4.1).

This notebook trains three variants **with the v10 recipe unchanged** (lr 2e-5, 6 epochs, batch 16, weighted EB/S2R sampler,
macro-F1 checkpoint selection, thresholds tuned on validation, test scored once) and changes only one thing at a time:

| run | what changes vs v10 |
|---|---|
| `aug` | extra **training-only** sentences (rule-based rewrites + Gemini paraphrases of EB / S2R sentences) |
| `ctx` | the model also sees the **previous sentence** of the same bug report |
| `aug_ctx` | both |

The v10 baseline is not retrained: its predictions (`baseline_probs.npz`, `baseline_results.json`) were produced locally from the
verified v10 checkpoint (SHA-256 `171562925ab6...`) and are compared against the new runs with a **paired bootstrap over bug reports**,
because a single run's score moves by a few points from chance alone (v10 vs v11, Table 5.3).

**Before running** - create a Kaggle dataset containing: `bug_reports_train.csv`, `bug_reports_val.csv`, `bug_reports_test.csv`,
`bug_reports_train_aug.csv` (from `augment_data.py`), `baseline_probs.npz` and `baseline_results.json`.
Attach it to this notebook, turn the **GPU on** and **Internet on** (the notebook downloads `roberta-large`).

In [ ]:
import glob, json, os, shutil, subprocess, sys, time

# ---- edit these -------------------------------------------------------------------------
RUNS = {                       # name -> train.py switches
    "aug":     ["--train-file", "bug_reports_train_aug.csv"],
    "ctx":     ["--context", "1"],
    "aug_ctx": ["--train-file", "bug_reports_train_aug.csv", "--context", "1"],
}
SEEDS  = [42]                  # add 43, 44 for more evidence (each extra seed repeats every run)
EPOCHS = 6                     # v10 used 6 (early stopping patience 2)
# -----------------------------------------------------------------------------------------

hits = glob.glob("/kaggle/input/**/bug_reports_val.csv", recursive=True)
assert hits, "Attach the dataset with bug_reports_*.csv to this notebook"
DATA_DIR = os.path.dirname(hits[0])
print("data dir:", DATA_DIR, sorted(os.listdir(DATA_DIR)))
os.makedirs("runs", exist_ok=True)
for f in ("baseline_probs.npz", "baseline_results.json"):      # v10 baseline for the comparison
    if os.path.exists(os.path.join(DATA_DIR, f)):
        shutil.copy(os.path.join(DATA_DIR, f), "runs")

In [ ]:
!pip -q install tabulate

In [ ]:
%%writefile common.py
"""
Shared pieces for Component 1 training and evaluation.

The model, dataset semantics and evaluation logic are identical to the v10 notebook
(Appendix, Code Listings 1-7). The only difference is dynamic padding: sentences are
padded to the longest one in the batch instead of to MAX_LEN. Padded positions are masked
by the attention mask, so predictions are unchanged; it just runs much faster.
"""

import hashlib

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, RobertaModel

MODEL_NAME = "roberta-large"
MAX_LEN = 128
LABELS = ["OB", "EB", "S2R"]
THRESHOLD_GRID = np.arange(0.1, 0.91, 0.05)


class MultiTaskRoberta(nn.Module):
    """Shared RoBERTa-Large encoder with a structure head (3 sigmoids) and a quality head (P(LOW))."""

    def __init__(self, model_name=MODEL_NAME, hidden_dropout=0.3, head_hidden=256, pretrained=True):
        super().__init__()
        if pretrained:
            self.encoder = RobertaModel.from_pretrained(model_name)
        else:  # architecture only, weights come from a checkpoint
            from transformers import RobertaConfig
            self.encoder = RobertaModel(RobertaConfig.from_pretrained(model_name))
        dim = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(hidden_dropout)
        self.struct_head = nn.Sequential(nn.Linear(dim, head_hidden), nn.GELU(), nn.Dropout(hidden_dropout), nn.Linear(head_hidden, 3))
        self.quality_head = nn.Sequential(nn.Linear(dim, head_hidden), nn.GELU(), nn.Dropout(hidden_dropout), nn.Linear(head_hidden, 1))

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(out.last_hidden_state[:, 0, :])
        return self.struct_head(pooled), self.quality_head(pooled).squeeze(-1)


def get_tokenizer():
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    # Guard against the tokenizer bug that invalidated earlier runs (Section 5.4.1).
    assert len(tok.tokenize("Thread leaks in Tomcat on hot-redeploy")) < 15, "TOKENIZER IS CHARACTER-SPLITTING"
    return tok


class BugReportDataset(Dataset):
    """
    One sentence per row. quality_label: LOW -> 1, HIGH -> 0, anything else is masked out of the
    quality loss. Rows with `quality_trusted == 0` (machine-generated paraphrases, whose quality
    label we cannot vouch for) are also masked out, but still train the structure head.
    """

    def __init__(self, df, ctx=None):
        self.texts = df["sentence_text"].astype(str).tolist()
        self.ctx = list(ctx) if ctx is not None else None
        self.struct = df[["is_OB", "is_EB", "is_S2R"]].astype("float32").values
        self.q_raw = df["quality_label"].tolist()
        self.trusted = df["quality_trusted"].tolist() if "quality_trusted" in df else [1] * len(df)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        q = self.q_raw[i]
        if q == "LOW" and self.trusted[i] != 0:
            ql, qm = 1.0, 1.0
        elif q == "HIGH" and self.trusted[i] != 0:
            ql, qm = 0.0, 1.0
        else:
            ql, qm = 0.0, 0.0
        return self.texts[i], (self.ctx[i] if self.ctx is not None else None), self.struct[i], ql, qm


def make_collate(tokenizer):
    def collate(batch):
        texts, ctx, struct, ql, qm = zip(*batch)
        if ctx[0] is None:
            enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        else:  # sentence first, the sentence before it as the second segment
            enc = tokenizer(list(texts), list(ctx), truncation="longest_first", max_length=MAX_LEN, padding=True, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "struct_labels": torch.tensor(np.stack(struct), dtype=torch.float32),
            "quality_label": torch.tensor(ql, dtype=torch.float32),
            "quality_mask": torch.tensor(qm, dtype=torch.float32),
        }
    return collate


def add_context(df):
    """
    The sentence immediately before each one in the same bug report ('' for the first sentence).
    Augmented rows (see augment_data.py) inherit the context of the sentence they were made from.
    """
    if "source_row" in df.columns and "aug_type" in df.columns:
        orig = df[df.aug_type == "original"]
        prev = orig.groupby("bug_id")["sentence_text"].shift(1).fillna("").astype(str)
        by_source = dict(zip(orig["source_row"], prev))
        return [by_source.get(int(r), "") if t != "original" else prev.loc[i]
                for i, r, t in zip(df.index, df["source_row"], df["aug_type"])]
    return df.groupby("bug_id")["sentence_text"].shift(1).fillna("").astype(str).tolist()


def make_loader(df, tokenizer, batch_size=32, shuffle=False, sampler=None, use_context=False):
    return DataLoader(BugReportDataset(df, add_context(df) if use_context else None), batch_size=batch_size, shuffle=shuffle and sampler is None,
                      sampler=sampler, collate_fn=make_collate(tokenizer))


@torch.no_grad()
def get_predictions(model, loader, device, amp=True):
    model.eval()
    sp, sl, qp, ql, qm = [], [], [], [], []
    for b in loader:
        with torch.autocast(device_type="cuda", enabled=amp and device.type == "cuda"):
            s_logits, q_logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
        sp.append(torch.sigmoid(s_logits.float()).cpu())
        qp.append(torch.sigmoid(q_logits.float()).cpu())
        sl.append(b["struct_labels"]); ql.append(b["quality_label"]); qm.append(b["quality_mask"])
    return (torch.cat(sp).numpy(), torch.cat(sl).numpy(), torch.cat(qp).numpy(),
            torch.cat(ql).numpy(), torch.cat(qm).numpy().astype(bool))


def prf(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return float(p), float(r), float(f)


def tune_thresholds(probs, labels):
    """Per-class threshold that maximises F1 on validation (same grid as the v10 notebook)."""
    out = []
    for i in range(probs.shape[1]):
        best_f, best_t = 0.0, 0.5
        for t in THRESHOLD_GRID:
            f = prf(labels[:, i], (probs[:, i] >= t).astype(int))[2]
            if f > best_f:
                best_f, best_t = f, float(t)
        out.append(best_t)
    return np.array(out)


def tune_quality_threshold(probs, labels, mask):
    best_f, best_t = 0.0, 0.5
    for t in THRESHOLD_GRID:
        f = prf(labels[mask], (probs[mask] >= t).astype(int))[2]
        if f > best_f:
            best_f, best_t = f, float(t)
    return best_t


def score(split, thresholds, q_threshold):
    """Per-class P/R/F1 + macro-F1 for the structure head, and P/R/F1 for quality, at fixed thresholds."""
    sp, sl, qp, ql, qm = split
    pred = (sp >= thresholds).astype(int)
    res = {}
    for i, name in enumerate(LABELS):
        p, r, f = prf(sl[:, i], pred[:, i])
        res[name] = {"precision": p, "recall": r, "f1": f}
    res["macro_f1"] = float(np.mean([res[n]["f1"] for n in LABELS]))
    p, r, f = prf(ql[qm], (qp[qm] >= q_threshold).astype(int))
    res["quality"] = {"precision": p, "recall": r, "f1": f}
    return res


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def load_csvs(data_dir):
    return (pd.read_csv(f"{data_dir}/bug_reports_train.csv"),
            pd.read_csv(f"{data_dir}/bug_reports_val.csv"),
            pd.read_csv(f"{data_dir}/bug_reports_test.csv"))

In [ ]:
%%writefile train.py
"""
Train Component 1 (multi-task RoBERTa-Large) - the v10 recipe, plus switches for the improvements.

    python train.py --data-dir <dir> --run-name baseline
    python train.py --data-dir <dir> --run-name aug      --train-file bug_reports_train_aug.csv
    python train.py --data-dir <dir> --run-name ctx      --context 1
    python train.py --data-dir <dir> --run-name aug_ctx  --train-file bug_reports_train_aug.csv --context 1

With no switches this is the v10 recipe: lr 2e-5, weight decay 0.05, 6 epochs, warmup 10%, batch 16,
AMP, grad-clip 1.0, EB/S2R weighted sampler, raw pos-weights for EB/S2R and sqrt-dampened OB,
checkpoint chosen by validation macro-F1 (patience 2), thresholds tuned on validation, test scored once.
"""

import argparse
import json
import os
import random
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import WeightedRandomSampler
from transformers import get_linear_schedule_with_warmup

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import (LABELS, MultiTaskRoberta, get_predictions, get_tokenizer, make_loader, score, sha256_of,
                    tune_quality_threshold, tune_thresholds)


def pos_weight_from_counts(n_total, n_pos, dampen=False, cap=None):
    raw = (n_total - n_pos) / max(n_pos, 1)
    w = np.sqrt(raw) if dampen else raw
    return min(w, cap) if cap is not None else w


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", required=True)
    ap.add_argument("--train-file", default="bug_reports_train.csv")
    ap.add_argument("--run-name", required=True)
    ap.add_argument("--out", default="runs")
    ap.add_argument("--model", default="roberta-large")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--epochs", type=int, default=6)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--weight-decay", type=float, default=0.05)
    ap.add_argument("--warmup", type=float, default=0.1)
    ap.add_argument("--patience", type=int, default=2)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--context", type=int, default=0, help="1 = also feed the previous sentence of the same bug report")
    ap.add_argument("--limit", type=int, default=0, help="smoke test: use only N rows per split")
    args = ap.parse_args()

    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(args.out, exist_ok=True)
    ckpt = os.path.join(args.out, f"{args.run_name}.pt")

    train_df = pd.read_csv(os.path.join(args.data_dir, args.train_file))
    val_df = pd.read_csv(os.path.join(args.data_dir, "bug_reports_val.csv"))
    test_df = pd.read_csv(os.path.join(args.data_dir, "bug_reports_test.csv"))
    if args.limit:
        rng = np.random.RandomState(0)
        keep = lambda d: d.iloc[np.sort(rng.choice(len(d), min(args.limit, len(d)), replace=False))].reset_index(drop=True)
        train_df, val_df, test_df = keep(train_df), keep(val_df), keep(test_df)
        for d in (val_df, test_df):  # context needs neighbours; in a smoke test they are simply absent
            pass
    n_aug = int((train_df.get("aug_type", pd.Series(["original"] * len(train_df))) != "original").sum())
    print(f"[{args.run_name}] train rows {len(train_df)} (augmented {n_aug}) | val {len(val_df)} | test {len(test_df)} | device {device} | context={args.context}")

    tok = get_tokenizer()
    # ----- sampler: oversample EB- and S2R-positive rows to balance (v10) -----
    eb = train_df["is_EB"].values == 1
    s2r = train_df["is_S2R"].values == 1
    w = np.ones(len(train_df))
    w = np.where(eb, np.maximum(w, (len(train_df) - eb.sum()) / max(eb.sum(), 1)), w)
    w = np.where(s2r, np.maximum(w, (len(train_df) - s2r.sum()) / max(s2r.sum(), 1)), w)
    sampler = WeightedRandomSampler(w, num_samples=len(train_df), replacement=True)
    train_loader = make_loader(train_df, tok, args.batch_size, sampler=sampler, use_context=bool(args.context))
    val_loader = make_loader(val_df, tok, 64, use_context=bool(args.context))
    test_loader = make_loader(test_df, tok, 64, use_context=bool(args.context))

    # ----- class weights (v10) -----
    n = len(train_df)
    struct_pos_weight = torch.tensor([
        pos_weight_from_counts(n, train_df.is_OB.sum(), dampen=True, cap=10.0),
        pos_weight_from_counts(n, train_df.is_EB.sum()),
        pos_weight_from_counts(n, train_df.is_S2R.sum()),
    ], dtype=torch.float32).to(device)
    trusted = train_df["quality_trusted"] != 0 if "quality_trusted" in train_df else pd.Series(True, index=train_df.index)
    structural = (train_df.is_OB == 1) | (train_df.is_EB == 1) | (train_df.is_S2R == 1)
    qc = train_df.loc[structural & trusted, "quality_label"].value_counts()
    quality_pos_weight = torch.tensor(qc.get("HIGH", 1) / max(qc.get("LOW", 1), 1), dtype=torch.float32).to(device)

    def combined_loss(s_logits, s_labels, q_logits, q_label, q_mask):
        s_loss = nn.BCEWithLogitsLoss(pos_weight=struct_pos_weight)(s_logits, s_labels)
        q_raw = nn.functional.binary_cross_entropy_with_logits(q_logits, q_label, pos_weight=quality_pos_weight, reduction="none")
        q_loss = (q_raw * q_mask).sum() / q_mask.sum().clamp(min=1.0)
        return s_loss + q_loss

    model = MultiTaskRoberta(args.model).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    total = len(train_loader) * args.epochs
    sched = get_linear_schedule_with_warmup(opt, int(total * args.warmup), total)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    history, best, bad, t0 = [], -1.0, 0, time.time()
    for epoch in range(1, args.epochs + 1):
        model.train()
        run_loss = 0.0
        for step, b in enumerate(train_loader, 1):
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=device.type == "cuda"):
                s_logits, q_logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                loss = combined_loss(s_logits.float(), b["struct_labels"].to(device), q_logits.float(),
                                     b["quality_label"].to(device), b["quality_mask"].to(device))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item()
            if step % 500 == 0:
                print(f"  epoch {epoch} step {step}/{len(train_loader)} loss {run_loss / step:.4f} ({(time.time() - t0) / 60:.1f} min)", flush=True)

        val = get_predictions(model, val_loader, device)
        m = score(val, np.array([0.5, 0.5, 0.5]), 0.5)  # selection criterion exactly as v10: macro-F1 at 0.5
        history.append({"epoch": epoch, "train_loss": run_loss / len(train_loader), "val_macro_f1_at_0.5": m["macro_f1"],
                        **{f"val_{c}_f1": m[c]["f1"] for c in LABELS}})
        print(f"epoch {epoch}/{args.epochs}  train loss {run_loss / len(train_loader):.4f}  val macro-F1 {m['macro_f1']:.3f}"
              f"  (OB {m['OB']['f1']:.3f} EB {m['EB']['f1']:.3f} S2R {m['S2R']['f1']:.3f})  {(time.time() - t0) / 60:.1f} min", flush=True)
        if m["macro_f1"] > best:
            best, bad = m["macro_f1"], 0
            torch.save(model.state_dict(), ckpt)
            print(f"  -> new best, saved {ckpt}", flush=True)
        else:
            bad += 1
            if bad >= args.patience:
                print(f"early stopping at epoch {epoch}")
                break

    # ----- thresholds on validation, test scored once -----
    model.load_state_dict(torch.load(ckpt, map_location=device))
    val = get_predictions(model, val_loader, device)
    test = get_predictions(model, test_loader, device)
    th, qt = tune_thresholds(val[0], val[1]), tune_quality_threshold(val[2], val[3], val[4])
    res = {
        "run": args.run_name, "config": vars(args), "train_rows": len(train_df), "augmented_rows": n_aug,
        "minutes": round((time.time() - t0) / 60, 1), "history": history, "sha256": sha256_of(ckpt),
        "thresholds": dict(zip(LABELS + ["QUALITY"], [*map(float, th), qt])),
        "val": score(val, th, qt), "test": score(test, th, qt),
    }
    json.dump(res, open(os.path.join(args.out, f"{args.run_name}_results.json"), "w"), indent=2)
    np.savez(os.path.join(args.out, f"{args.run_name}_probs.npz"),
             val_struct=val[0], val_labels=val[1], val_q=val[2], val_ql=val[3], val_qm=val[4],
             test_struct=test[0], test_labels=test[1], test_q=test[2], test_ql=test[3], test_qm=test[4],
             thresholds=th, q_threshold=qt)
    print(f"\n=== {args.run_name}: TEST (thresholds tuned on validation) ===")
    for c in LABELS:
        r = res["test"][c]
        print(f"  {c:<3} P={r['precision']:.3f} R={r['recall']:.3f} F1={r['f1']:.3f} (threshold {res['thresholds'][c]:.2f})")
    q = res["test"]["quality"]
    print(f"  Structure macro-F1: {res['test']['macro_f1']:.3f}   |  Quality F1 {q['f1']:.3f}  |  SHA-256 {res['sha256']}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile compare_runs.py
"""
Compare training runs fairly.

    python compare_runs.py --runs-dir runs --test-csv <bug_reports_test.csv> [--baseline baseline]

With only ~190 Expected-Behaviour sentences in the test set, a single run's score moves by a few
points from chance alone (the v10 vs v11 result in Table 5.3 is an example). So this script reports:
  * every run's TEST macro-F1 with a 95% bootstrap confidence interval (resampling whole bug
    reports, because sentences from one report are not independent), and
  * for each run, the paired difference to the baseline run with its own interval and the
    probability that it is really better.
It also scores an ensemble (average of the runs' probabilities, thresholds re-tuned on validation).
"""

import argparse
import glob
import json
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import LABELS, score, tune_quality_threshold, tune_thresholds

N_BOOT = 2000


def load_run(path):
    name = os.path.basename(path).replace("_probs.npz", "")
    z = np.load(path)
    meta = json.load(open(path.replace("_probs.npz", "_results.json")))
    return name, z, meta


def per_bug_counts(probs, labels, thresholds, bug_ids):
    """tp/fp/fn per bug report for each class -> array [n_bugs, 3 classes, 3]."""
    pred = (probs >= thresholds).astype(int)
    codes, uniq = pd.factorize(bug_ids)
    out = np.zeros((len(uniq), 3, 3))
    for c in range(3):
        for k, m in enumerate(((pred[:, c] == 1) & (labels[:, c] == 1), (pred[:, c] == 1) & (labels[:, c] == 0), (pred[:, c] == 0) & (labels[:, c] == 1))):
            np.add.at(out[:, c, k], codes, m.astype(float))
    return out


def macro_f1_from_counts(counts):  # counts [..., 3 classes, 3]
    tp, fp, fn = counts[..., 0], counts[..., 1], counts[..., 2]
    f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9)
    return f1.mean(axis=-1), f1


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs-dir", default="runs")
    ap.add_argument("--test-csv", required=True)
    ap.add_argument("--baseline", default="baseline")
    ap.add_argument("--out", default=None)
    args = ap.parse_args()

    test = pd.read_csv(args.test_csv)
    runs = [load_run(p) for p in sorted(glob.glob(os.path.join(args.runs_dir, "*_probs.npz")))]
    if not runs:
        sys.exit("no *_probs.npz files found")
    rng = np.random.RandomState(0)
    n_bugs = test["bug_id"].nunique()
    boot_idx = rng.randint(0, n_bugs, size=(N_BOOT, n_bugs))

    rows, boots = [], {}
    for name, z, meta in runs:
        th = z["thresholds"]
        counts = per_bug_counts(z["test_struct"], z["test_labels"], th, test["bug_id"].values)
        boot_counts = counts[boot_idx].sum(axis=1)  # [N_BOOT, 3, 3]
        macro, per_class = macro_f1_from_counts(boot_counts)
        boots[name] = (macro, per_class)
        point = meta["test"]
        rows.append({
            "run": name, "macro_f1": point["macro_f1"], "ci95": f"{np.percentile(macro, 2.5):.3f}-{np.percentile(macro, 97.5):.3f}",
            "OB": point["OB"]["f1"], "EB": point["EB"]["f1"], "S2R": point["S2R"]["f1"], "quality_f1": point["quality"]["f1"],
            "val_macro_f1": meta["val"]["macro_f1"], "minutes": meta.get("minutes"),
        })

    # Ensemble of all runs (probability average, thresholds tuned on validation only)
    if len(runs) > 1:
        avg = lambda key: np.mean([z[key] for _, z, _ in runs], axis=0)
        val = (avg("val_struct"), runs[0][1]["val_labels"], avg("val_q"), runs[0][1]["val_ql"], runs[0][1]["val_qm"])
        tst = (avg("test_struct"), runs[0][1]["test_labels"], avg("test_q"), runs[0][1]["test_ql"], runs[0][1]["test_qm"])
        th, qt = tune_thresholds(val[0], val[1]), tune_quality_threshold(val[2], val[3], val[4])
        s = score(tst, th, qt)
        counts = per_bug_counts(tst[0], tst[1], th, test["bug_id"].values)
        macro, per_class = macro_f1_from_counts(counts[boot_idx].sum(axis=1))
        boots["ENSEMBLE"] = (macro, per_class)
        rows.append({"run": "ENSEMBLE", "macro_f1": s["macro_f1"], "ci95": f"{np.percentile(macro, 2.5):.3f}-{np.percentile(macro, 97.5):.3f}",
                     "OB": s["OB"]["f1"], "EB": s["EB"]["f1"], "S2R": s["S2R"]["f1"], "quality_f1": s["quality"]["f1"],
                     "val_macro_f1": score(val, th, qt)["macro_f1"], "minutes": None})

    table = pd.DataFrame(rows)
    lines = ["# Run comparison (held-out TEST, thresholds tuned on validation)", "", table.round(3).to_markdown(index=False), ""]

    if args.baseline in boots:
        lines += [f"## Difference to `{args.baseline}` (paired bootstrap over bug reports, {N_BOOT} resamples)", "",
                  "| run | macro-F1 gain | 95% interval | P(better) | EB gain | S2R gain |", "|---|---|---|---|---|---|"]
        b_macro, b_pc = boots[args.baseline]
        for name, (m, pc) in boots.items():
            if name == args.baseline:
                continue
            d = m - b_macro
            lines.append(f"| {name} | {d.mean():+.3f} | {np.percentile(d, 2.5):+.3f} to {np.percentile(d, 97.5):+.3f} | {(d > 0).mean():.0%} | "
                         f"{(pc[:, 1] - b_pc[:, 1]).mean():+.3f} | {(pc[:, 2] - b_pc[:, 2]).mean():+.3f} |")
        lines += ["", "Rule of thumb: call a run better only when the interval excludes zero (P(better) above about 95%)."]

    text = "\n".join(lines)
    print(text)
    open(args.out or os.path.join(args.runs_dir, "comparison.md"), "w", encoding="utf8").write(text)


if __name__ == "__main__":
    main()

In [ ]:
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Turn the GPU on in the notebook settings"
import pandas as pd
for n in ("train", "train_aug", "val", "test"):
    p = os.path.join(DATA_DIR, f"bug_reports_{n}.csv")
    if os.path.exists(p):
        d = pd.read_csv(p); print(f"{n:<10} {len(d):>7} rows | EB {int(d.is_EB.sum()):>5} S2R {int(d.is_S2R.sum()):>5}")

## Train
Each run saves `runs/<name>.pt`, `<name>_results.json` and `<name>_probs.npz`. Finished runs are skipped, so the cell is safe to re-run after a disconnect.

In [ ]:
for seed in SEEDS:
    for name, switches in RUNS.items():
        run = name if seed == SEEDS[0] and len(SEEDS) == 1 else f"{name}_s{seed}"
        if os.path.exists(f"runs/{run}_results.json"):
            print("skip (done):", run); continue
        cmd = [sys.executable, "-u", "train.py", "--data-dir", DATA_DIR, "--run-name", run, "--seed", str(seed), "--epochs", str(EPOCHS), *switches]
        print("\n" + "=" * 100 + "\n" + " ".join(cmd) + "\n" + "=" * 100, flush=True)
        t0 = time.time()
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            if "Warning" not in line:
                print(line, end="", flush=True)
        p.wait()
        print(f"[{run}] finished with code {p.returncode} in {(time.time() - t0) / 60:.1f} min")

## Compare with v10
Read the *P(better)* column: only treat a run as an improvement when its interval excludes zero.

In [ ]:
out = subprocess.run([sys.executable, "compare_runs.py", "--runs-dir", "runs", "--test-csv", os.path.join(DATA_DIR, "bug_reports_test.csv"), "--baseline", "baseline"],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)

## What to do with the result

1. Download `runs/` (at least `*_results.json`, `*_probs.npz`, `comparison.md` and the `.pt` of the best run) from the notebook output.
2. Re-verify the best checkpoint locally with `evaluate_checkpoint.py` (same step as Code Listing 7) and check its SHA-256 matches `*_results.json`.
3. Deploy only if it beats v10 with a clear margin (interval excludes zero) **and** does not lose on the class it was not meant to help.
   If `ctx` or `aug_ctx` wins, the service also has to feed the previous sentence at inference time (a small change in `app.py`).
4. Report negative results as honestly as v11 was reported: an idea that does not clear the noise is still a finding.